# SAFE-Alert — Ablation Study (9 cells riêng)

Mỗi cell = 1 variant. Chạy xong cell nào là có kết quả ngay.

**Setup:** Add dataset `run-baseline-v2` → GPU T4 x2 → Chạy cell Setup trước.

In [ ]:
import os, sys, glob, subprocess, json, zipfile
import torch

WORK_DIR   = '/kaggle/working'
AI_SERVICE = '/kaggle/input/datasets/minhquan0706/run-baseline-v2/ai-service'
DATA_V2    = AI_SERVICE + '/training_data/v2'

assert os.path.exists(AI_SERVICE), f'AI_SERVICE not found: {AI_SERVICE}'
assert os.path.exists(DATA_V2),    f'DATA_V2 not found: {DATA_V2}'

PIPELINES = os.path.join(AI_SERVICE, 'app', 'v2', 'pipelines')
for p in [PIPELINES, os.path.join(AI_SERVICE,'app','v2'), AI_SERVICE]:
    sys.path.insert(0, p)
os.chdir(AI_SERVICE)

ARTIFACT_DIR = os.path.join(WORK_DIR, 'ablation')
os.makedirs(ARTIFACT_DIR, exist_ok=True)

SCRIPT = os.path.join(PIPELINES, 'run_ablation.py')
BASE_CMD = (
    f'python -u {SCRIPT}'
    f' --symbol BTCUSDT --horizon 1h --epochs 40 --batch_size 32'
    f' --data_path {DATA_V2} --embeddings_path {DATA_V2}'
    f' --artifact_dir {ARTIFACT_DIR}'
)

def run_variant(name):
    out = os.path.join(ARTIFACT_DIR, f'result_{name.replace("/","_")}.json')
    cmd = f'{BASE_CMD} --variants {name} --output {out}'
    print(f'>>> [{name}]', flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if os.path.exists(out):
        with open(out) as f:
            r = json.load(f)
        m = r.get(name, {})
        print(f'DONE {name}: F1={m.get("macro_f1",0):.3f}  Sharpe={m.get("alert_sharpe",0):.3f}', flush=True)
    else:
        print(f'ERROR: {out} not created', flush=True)
    return out

!pip install -q ta==0.11.0 vaderSentiment==3.3.2
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('Setup done. Chạy từng cell variant bên dưới.')

In [ ]:
# VARIANT 1: full_model (~40 phút) — baseline để so sánh
run_variant('full_model')

In [ ]:
# VARIANT 2: w/o_selective_news (~35 phút)
run_variant('w/o_selective_news')

In [ ]:
# VARIANT 3: w/o_factor (~35 phút)
run_variant('w/o_factor')

In [ ]:
# VARIANT 4: w/o_market (~30 phút)
run_variant('w/o_market')

In [ ]:
# VARIANT 5: w/o_confidence (~35 phút)
run_variant('w/o_confidence')

In [ ]:
# VARIANT 6: w/o_faithfulness (~35 phút)
run_variant('w/o_faithfulness')

In [ ]:
# VARIANT 7: w/o_horizon (~35 phút)
run_variant('w/o_horizon')

In [ ]:
# VARIANT 8: w/o_lrisk (~35 phút)
run_variant('w/o_lrisk')

In [ ]:
# VARIANT 9: w/o_bar_sequences (~35 phút)
run_variant('w/o_bar_sequences')

In [ ]:
# ===== MERGE tất cả → ablation_results.json =====
ALL_VARIANTS = [
    'full_model','w/o_selective_news','w/o_factor','w/o_market',
    'w/o_confidence','w/o_faithfulness','w/o_horizon','w/o_lrisk','w/o_bar_sequences'
]

merged = {}
missing = []
for name in ALL_VARIANTS:
    fname = name.replace('/', '_')
    path = os.path.join(ARTIFACT_DIR, f'result_{fname}.json')
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if name in data:
            merged[name] = data[name]
            f1 = data[name].get('macro_f1', 0)
            sh = data[name].get('alert_sharpe', 0)
            print(f'[OK] {name:30s} F1={f1:.3f}  Sharpe={sh:.3f}')
    else:
        missing.append(name)
        print(f'[--] {name:30s} chưa chạy')

merged['_meta'] = {'protocol':'single_split_70_15_15','epochs':40,'n_variants_run':len(merged)}
out = os.path.join(ARTIFACT_DIR, 'ablation_results.json')
with open(out, 'w') as f:
    json.dump(merged, f, indent=2)
print(f'\nMerged {len(merged)-1}/9 variants → {out}')
if missing:
    print(f'Chưa có: {missing}')

# Bảng so sánh
full_f1 = merged.get('full_model', {}).get('macro_f1', 0)
print(f'\n  {"Variant":30s}  {"F1":>6}  {"ΔF1":>7}  {"Sharpe":>7}  {"MCC":>6}')
print('  ' + '-'*65)
for name in ALL_VARIANTS:
    if name not in merged: continue
    m   = merged[name]
    f1  = m.get('macro_f1', 0)
    sh  = m.get('alert_sharpe', 0)
    mcc = m.get('mcc', 0)
    df1 = f1 - full_f1
    print(f'  {name:30s}  {f1:.3f}  {df1:+7.3f}  {sh:7.3f}  {mcc:6.3f}')

zip_all = os.path.join(WORK_DIR, 'ablation_all.zip')
with zipfile.ZipFile(zip_all, 'w') as zf:
    zf.write(out, 'ablation_results.json')
    for name in ALL_VARIANTS:
        p = os.path.join(ARTIFACT_DIR, f'result_{name.replace("/","_")}.json')
        if os.path.exists(p): zf.write(p, os.path.basename(p))
print(f'\nDownload: {zip_all} ({os.path.getsize(zip_all)/1e3:.1f} KB)')